# NovaTech AI Knowledge Assistant (RAG + Buffer Memory)

**Pipeline:** Data (PDF + Google Sheets) → Embedding & Indexing (HuggingFace + FAISS) → RAG Question Answering (Qwen) → Buffer Memory → Testing

Before running: upload to Colab (`/content`) the files `NovaTech_Employee_Handbook.pdf`, `NovaTech_Employees.csv`, `NovaTech_Departments.csv`, `NovaTech_Test_Questions.xlsx`, and use a **T4 GPU** runtime.

## 0. Imports

In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers accelerate pypdf pandas openpyxl

In [ ]:
import re
import numpy as np
import pandas as pd
import torch
import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

## 1. Configuration

For **Google Sheets**, share the sheet as *Anyone with the link → Viewer* and use the export URL of each tab:
`https://docs.google.com/spreadsheets/d/<SHEET_ID>/export?format=csv&gid=<TAB_GID>`
Local CSV/XLSX paths work too (the loader accepts both).

In [ ]:
PDF_PATH = "/content/NovaTech_Employee_Handbook.pdf"

# name -> Google Sheets export URL  OR  local .csv / .xlsx path
SHEET_SOURCES = {
    "Employees":   "/content/NovaTech_Employees.csv",
    "Departments": "/content/NovaTech_Departments.csv",
}

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_NAME    = "Qwen/Qwen2.5-3B-Instruct"   # fits a free T4. Use "Qwen/Qwen2.5-7B-Instruct" on a bigger GPU
K_PDF, K_SHEET = 3, 3                      # chunks retrieved from each source

## 2. Data Integration (PDF + Sheets → text)

In [ ]:
def load_pdf(path):
    text = ""
    for page in PdfReader(path).pages:
        text += (page.extract_text() or "") + "\n"
    return text


def load_sheet(src):
    if src.endswith(".xlsx"):
        return pd.read_excel(src)
    return pd.read_csv(src)          # also works with a Google Sheets export URL


def row_to_text(row, sheet_name):
    """Turn one sheet row into a readable sentence (better for semantic search than raw 'col: value' pairs)."""
    r = {k: ("" if pd.isna(v) else str(v).strip()) for k, v in row.items()}

    if {"Name", "Job_Title", "Department"} <= r.keys():            # Employees
        hire = pd.to_datetime(r["Hire_Date"])
        years = int((pd.Timestamp.today() - hire).days // 365.25)
        text = (f"{r['Name']} (Employee ID {r['Employee_ID']}) works in the {r['Department']} department "
                f"as {r['Job_Title']}. ")
        if r["Manager"]:
            text += f"Manager: {r['Manager']}. "
        text += (f"Hired on {hire.date()} ({years} years of service). Office: {r['Office']}. "
                 f"Work mode: {r['Work_Mode']}. Salary grade: {r['Salary_Grade']}. "
                 f"Annual leave balance: {r['Annual_Leave_Balance_Days']} days. Email: {r['Email']}.")
        return text

    if "Department_Head" in r:                                       # Departments
        return (f"The {r['Department']} department is headed by {r['Department_Head']}. "
                f"Location: {r['Location']}. Cost center: {r['Cost_Center']}. Phone extension: {r['Extension']}.")

    return f"{sheet_name}: " + " | ".join(f"{k}: {v}" for k, v in r.items() if v)   # any other sheet

In [ ]:
def chunk_pdf(text, size=120, overlap=20):
    """Split by numbered section ("3. Leave Policy"), then by words. Every chunk keeps its section title."""
    text = re.sub(r"[ \t]+", " ", text)
    sections = re.split(r"\n(?=\d{1,2}\. [A-Z])", text)
    chunks = []
    for sec in sections:
        sec = sec.strip()
        if not sec:
            continue
        title = sec.split("\n")[0].strip()
        words = sec.split()
        for i in range(0, len(words), size - overlap):
            piece = " ".join(words[i:i + size])
            if i > 0:
                piece = f"[{title}] " + piece
            chunks.append(piece)
            if i + size >= len(words):
                break
    return chunks


pdf_text   = load_pdf(PDF_PATH)
pdf_items  = [{"text": c, "source": "Handbook PDF"} for c in chunk_pdf(pdf_text)]

sheet_items = []
for name, src in SHEET_SOURCES.items():
    df = load_sheet(src)
    print(f"Sheet '{name}': {len(df)} rows, columns = {list(df.columns)}")
    for _, row in df.iterrows():
        sheet_items.append({"text": row_to_text(row, name), "source": f"Sheet: {name}"})

print(f"\nPDF chunks   : {len(pdf_items)}")
print(f"Sheet chunks : {len(sheet_items)}")
print("\nSample PDF chunk  :", pdf_items[3]["text"][:300])
print("\nSample sheet chunk:", sheet_items[2]["text"])

## 3. Embedding & Indexing (HuggingFace + FAISS)
Two FAISS indexes (PDF and Sheets). Retrieving from **each source separately** guarantees that questions mixing both sources (e.g. *"What bonus does Sara Hassan get?"*: her grade is in the sheet, the bonus table is in the PDF) get context from both.

In [ ]:
embed_model = SentenceTransformer(EMBED_MODEL)


def build_index(items):
    emb = embed_model.encode([i["text"] for i in items], convert_to_numpy=True,
                             normalize_embeddings=True, show_progress_bar=False).astype("float32")
    index = faiss.IndexFlatL2(emb.shape[1])
    index.add(emb)
    return index


def search_index(query, index, items, k=3):
    q = embed_model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    _, idx = index.search(q, min(k, len(items)))
    return [items[i] for i in idx[0] if i >= 0]


pdf_index   = build_index(pdf_items)
sheet_index = build_index(sheet_items)
print(f"FAISS ready: {pdf_index.ntotal} PDF vectors, {sheet_index.ntotal} sheet vectors")


# --- exact name lookup: MiniLM is weak with names, so if an employee's name appears in the question,
#     that employee's row is always included.
from collections import Counter
for it in sheet_items:
    if " (Employee ID" in it["text"]:
        it["full"] = it["text"].split(" (Employee ID")[0].lower()
first_counts = Counter(it["full"].split()[0] for it in sheet_items if "full" in it)
for it in sheet_items:
    if "full" in it and first_counts[it["full"].split()[0]] == 1:
        it["first"] = it["full"].split()[0]          # first name usable only if unique


def name_lookup(query):
    q = " " + re.sub(r"[^\w\s]", " ", query.lower()) + " "
    full = [it for it in sheet_items if "full" in it and f" {it['full']} " in q]
    if full:
        return full
    return [it for it in sheet_items if "first" in it and f" {it['first']} " in q]


def retrieve(query):
    hits = name_lookup(query)[:3]
    for h in search_index(query, pdf_index, pdf_items, K_PDF) + search_index(query, sheet_index, sheet_items, K_SHEET):
        if h not in hits:
            hits.append(h)
    return hits

## 4. Load the LLM (Qwen)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"
)
print(f"{LLM_NAME} ready on {device}")


def generate(messages, max_new_tokens=300):
    """messages = [{'role': 'system'|'user'|'assistant', 'content': ...}] → answer text.
    Greedy decoding (no sampling) → stable, factual answers for RAG."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)      # no truncation: never cut the context
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

## 5. Buffer Memory (full chat history)
Every turn is stored as `{"user", "assistant"}` and the **whole history** is sent to the model with each new question.

In [ ]:
conversation_history = []      # the buffer memory


def show_history():
    print("=" * 60, "\nFULL CONVERSATION HISTORY\n" + "=" * 60)
    for i, t in enumerate(conversation_history, 1):
        print(f"Turn {i}:\n  User     : {t['user']}\n  Assistant: {t['assistant']}\n")

## 6. RAG Pipeline with Memory
1. **Rewrite** the question using the history, so a follow-up like *"What is his grade?"* becomes *"What is Omar Khaled's salary grade?"* (the retriever has no memory, so it needs a standalone question).
2. **Retrieve** from the PDF and the Sheets.
3. **Generate** the answer with the context + the full chat history.
4. **Store** the turn in memory.

In [ ]:
SYSTEM_PROMPT = (
    "You are NovaTech's HR knowledge assistant. Be concise and follow the user's formatting requests "
    "(for example 'short answers').\n"
    "Rules:\n"
    "1. For questions about NovaTech, answer ONLY from the provided context (employee records and handbook). "
    "Combine facts from several context items when needed (e.g. an employee's salary grade comes from the "
    "employee records, the bonus for that grade comes from the handbook).\n"
    "2. For questions about the conversation itself (what the user said or asked earlier, the user's name, ...), "
    "answer from the chat history and ignore the context.\n"
    "3. If the user tells you something about themselves or gives you an instruction, just acknowledge it briefly.\n"
    "4. If a NovaTech question cannot be answered from the context, reply exactly: "
    "\"I couldn't find this information in the knowledge base.\""
)


def rewrite_question(history, question, last_n=3):
    """Make a follow-up standalone for the retriever. Uses only the last few turns (the memory itself stays full)."""
    if not history:
        return question
    hist = "\n".join(f"User: {t['user']}\nAssistant: {t['assistant'][:300]}" for t in history[-last_n:])
    messages = [
        {"role": "system", "content":
            "Rewrite the user's latest message as a standalone question for a search engine. "
            "Replace pronouns and vague references (he, she, it, they, that, this) with the exact name or topic "
            "they refer to in the MOST RECENT turn of the chat history. "
            "If the message is already self-contained, is not a question (a statement or an instruction), or is about "
            "the conversation itself (e.g. 'what was my first question?', 'what is my name?'), return it EXACTLY unchanged. "
            "Never answer it. Output one line only."},
        {"role": "user", "content": f"Chat history:\n{hist}\n\nLatest message: {question}\n\nStandalone question:"},
    ]
    out = generate(messages, max_new_tokens=60).strip().split("\n")[0].strip()
    return out if 0 < len(out) <= 3 * len(question) + 80 else question      # sanity check


def build_messages(history, question, standalone, hits):
    context = "\n".join(f"[{h['source']}] {h['text']}" for h in hits)
    final = f"Context:\n{context}\n\nQuestion: {question}"
    if standalone.strip() != question.strip():
        final += f"\n(Standalone version: {standalone})"
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for t in history:                                           # full buffer memory
        messages.append({"role": "user", "content": t["user"]})
        messages.append({"role": "assistant", "content": t["assistant"]})
    messages.append({"role": "user", "content": final})
    return messages


def ask(question, show_sources=False):
    standalone = rewrite_question(conversation_history, question)
    hits = retrieve(standalone)
    answer = generate(build_messages(conversation_history, question, standalone, hits))
    conversation_history.append({"user": question, "assistant": answer})
    if show_sources:
        print(f"   [standalone question] {standalone}")
        for h in hits:
            print(f"   [{h['source']}] {h['text'][:110]}...")
    return answer


def show_chat(q, show_sources=False):
    print(f"User     : {q}")
    print(f"Assistant: {ask(q, show_sources)}\n")

## 7. Demo

In [ ]:
conversation_history.clear()
show_chat("Who is the head of the Data & AI department?", show_sources=True)
show_chat("What is his salary grade?", show_sources=True)
show_chat("And what bonus percentage does that grade get?", show_sources=True)
show_chat("What was my first question?")
show_chat("What is NovaTech's stock price?")

In [ ]:
show_history()

## 8. Testing & Evaluation
Runs every question in `NovaTech_Test_Questions.xlsx`. Memory is reset before each question **except** questions marked *"after previous"* (they continue the same conversation).

Scoring (first pass, review manually afterwards):
- **Similarity**: cosine similarity between the expected and the actual answer (relevance).
- **Auto verdict**: ✅ if similarity ≥ 0.55 **and** every number in the expected answer appears in the answer. For out-of-scope questions ✅ only if the assistant says it couldn't find the information (no hallucination).

In [ ]:
TEST_FILE = "/content/NovaTech_Test_Questions.xlsx"
NOT_FOUND = "couldn't find"


def numbers_in(s):
    return [n.replace(",", "") for n in re.findall(r"\d[\d,\.]*%?", str(s))]


def evaluate(row):
    ans, exp = row["Actual_Answer"], row["Expected_Answer"]
    if row["Category"].startswith("Out of scope"):
        return 1.0, NOT_FOUND in ans.lower()
    e = embed_model.encode([exp, ans], normalize_embeddings=True)
    sim = float(np.dot(e[0], e[1]))
    nums_ok = all(n in ans.replace(",", "") for n in numbers_in(exp))
    return sim, (sim >= 0.55 and nums_ok)


tests = pd.read_excel(TEST_FILE)
tests["Actual_Answer"] = ""
tests["Similarity"] = 0.0
tests["Auto_Verdict"] = ""

for i, row in tests.iterrows():
    if "after previous" not in row["Category"]:
        conversation_history.clear()
    tests.at[i, "Actual_Answer"] = ask(row["Question"])
    sim, ok = evaluate(tests.loc[i])
    tests.at[i, "Similarity"] = round(sim, 3)
    tests.at[i, "Auto_Verdict"] = "✅" if ok else "❌"
    print(f"{row['ID']:>2} {tests.at[i, 'Auto_Verdict']}  {row['Question']}\n      → {tests.at[i, 'Actual_Answer'][:150]}")

conversation_history.clear()

In [ ]:
tests["Group"] = tests["Category"].str.split(" \\(").str[0]
summary = tests.groupby("Group")["Auto_Verdict"].apply(lambda s: f"{(s == '✅').sum()}/{len(s)}")
print(summary.to_string())
print(f"\nOverall: {(tests['Auto_Verdict'] == '✅').sum()}/{len(tests)} "
      f"= {(tests['Auto_Verdict'] == '✅').mean():.0%}   |   Avg similarity: {tests['Similarity'].mean():.2f}")

tests.drop(columns=["Group"]).to_excel("/content/test_results.xlsx", index=False)
print("Saved → /content/test_results.xlsx  (review the ❌ rows manually: a correct answer phrased differently can be marked ❌)")

## 9. Interactive Chat

In [ ]:
conversation_history.clear()
print("Chat started! Commands: 'quit' to stop, 'history' to see memory, 'reset' to clear memory.\n")

while True:
    user_input = input("You: ").strip()
    if user_input.lower() == "quit":
        print("Chat ended.")
        break
    if user_input.lower() == "history":
        show_history()
        continue
    if user_input.lower() == "reset":
        conversation_history.clear()
        print("Memory cleared.\n")
        continue
    if not user_input:
        continue
    print(f"Assistant: {ask(user_input)}\n")